In [1]:
import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re
import json






/tmp/ipykernel_3739701/2431725470.py:19: DeprecationWarning: hijri-converter is deprecated. Use 'hijridate' instead: pip install hijridate==2.3.0
  from hijri_converter import Hijri
/home/alotaime/miniconda3/envs/cs323/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
from config import MODEL_ID, DEVICE, GMAIL_EMAIL, APP_PASSWORD , CALENDAR_ID , SERVICE_ACCOUNT_FILE , DEVICE , SCOPES , IMAP_SERVER , SMTP_SERVER , tokenizer , model , llm_pipeline

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Device set to use cuda:0
The model 'OptimizedModule' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoForCausal

In [3]:
from processor import FileProcessor
from rag_engine import find_parent_large_chunks , initialize_fixed_knowledge , regex_split_documents , get_fixed_context , get_attachment_context
from services import safe_decode , html_to_text , send_reply , check_new_emails , update_email_history , create_draft , sync_sent_emails_to_history 
from All_intents import get_current_context , arabic_to_english_numbers , calculate_next_weekday , calculate_next_hijri , process_gregorian_date , process_hijri_date , extract_appointment_info , generate_llm_response , handle_calendar_intent , get_isolated_knowledge_context , handle_general_inquiries , handle_administrative_procedures , handle_administrative_procedures , handle_hr_procedures , rag_answer_email , handle_attachment_special_requests , handle_User_Requests , handle_administrative_procedures_2
from RulesForIntents import RULES_DIR , load_intent_rules , deduce_administrative_rule , update_intent_rules_file ,    retrieve_relevant_rules


from SmartRuleRetrieval import get_relevant_rule_for_intent

In [4]:
def classify_email_intent(email_body, sender_info):
    """
    محلل النية المركزي: يقوم بتصنيف الإيميل إلى فئة محددة لتوجيهه للدالة المناسبة مباشرة.
    """
    system_prompt = """
    أنت نظام خبير في تصنيف المراسلات الإدارية. مهمتك هي قراءة الإيميل وتصنيفه بدقة إلى فئة واحدة فقط من الفئات التالية:
    1. "CALENDAR": لطلبات تحديد أو تأكيد المواعيد والحضور صراحةً.
    2. "HR": لطلبات الموظفين الشخصية (إجازات، تدريب، دراسة).
    3. "ADMIN": إذا كان المرسل (مدير/شركة خدمات) يطلب التزاماً مالياً أو سداد فواتير.
    4. "INQUIRY": للاستفسارات عن الأنظمة، اللوائح، أو الإجراءات (وليس الصيانة).
    5. "User_Requests": (أولوية قصوى) لأي طلب صريح "لكتابة إيميل" أو "بحث في المرفقات" مهما كان الموضوع.
    6. "RAG_MAINTENANCE": لبلاغات الصيانة، الإصلاح، أو أي موضوع عام آخر يحتاج بحثاً ولا يندرج أعلاه.
    7. "ADMIN_2": للمراسلات المتعلقة بـ "طرح" المشاريع، "منصة اعتماد"، "جداول الكميات"، أو طلبات ترميم المباني الكاملة.
    أعد النتيجة بصيغة JSON فقط:
    {"intent": "الفئة_هنا"}
    """
    
    prompt = f"المرسل: {sender_info}\nنص الإيميل: {email_body}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True , enable_thinking=False)
        outputs = llm_pipeline(text_input, temperature=0.01) # حرارة منخفضة جداً للالتزام بالتصنيف
        raw_output = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group()).get("intent", "RAG_MAINTENANCE")
    except Exception as e:
        print(f"⚠ خطأ في تحليل النية المركزي: {e}")
    
    return "RAG_MAINTENANCE" 


In [5]:

def log_draft_for_evaluation(msg_id, sender, intent, incoming_body, bot_draft):
    """
    حفظ بيانات المسودة في ملف إكسل لغرض التقييم لاحقاً.
    """
    file_path = "evaluation_data.xlsx"
    new_data = {
        "Date": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
        "Message-ID": [msg_id],
        "Sender": [sender],
        "Intent": [intent],
        "Incoming Body": [incoming_body],
        "Bot Draft": [bot_draft],
        "Ground Truth": [""]  # يترك فارغاً لتقوم أنت بتعبئته
    }
    
    df_new = pd.DataFrame(new_data)
    
    try:
        if os.path.exists(file_path):
            df_existing = pd.read_excel(file_path)
            df_final = pd.concat([df_existing, df_new], ignore_index=True)
            # منع التكرار بناءً على Message-ID
            df_final.drop_duplicates(subset=['Message-ID'], keep='first', inplace=True)
            df_final.to_excel(file_path, index=False)
        else:
            df_new.to_excel(file_path, index=False)
        print(f"📊 تم تسجيل المسودة في ملف التقييم (ID: {msg_id})")
    except Exception as e:
        print(f"⚠ خطأ في تحديث ملف الإكسل: {e}")
    

In [ ]:
def run_auto_bot():
    print("🚀 جاري تشغيل بوت الرد الآلي المطور (نظام التقييم والتعلم الذاتي)...")
    
    # إعداد النماذج والمعالجات
    embed_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3", 
        model_kwargs={'device': DEVICE}
    )
    processor = FileProcessor()
    
    # البناء الأولي للقاعدة المعرفية الثابتة
    kb = initialize_fixed_knowledge(processor, embed_model)
    if not kb:
        print("❌ فشل بناء القاعدة المعرفية الثابتة.")
        return
    
    print("✅ تم تفعيل النظام بالكامل. البوت في وضع الاستعداد...")
    
    while True:
        try:
            # 1. فحص البريد الوارد ومعالجته
            new_emails = check_new_emails(processor)
            
            for sender, subject, body, attachments, msg_id in new_emails:
                print(f"\n📩 إيميل جديد من: {sender}")
                
                # تصنيف النية (Intent Classification)
                intent = classify_email_intent(body, sender)
                print(f"🎯 النية المكتشفة: {intent}")

                # تسجيل التاريخ المحلي في ملف النص
                update_email_history("إيميل وارد", body, msg_id=msg_id, intent=intent)
                
                reply_text = None

                # أ. معالجة المرفقات (إن وجدت)
                if attachments:
                    reply_text = handle_attachment_special_requests(attachments, body, processor, embed_model, sender)

                # ب. معالجة النصوص حسب النية إذا لم يتم الرد عبر المرفقات
                if not reply_text:
                    if intent == "CALENDAR":
                        reply_text = handle_calendar_intent(body, sender)
                    elif intent == "HR":
                        reply_text = handle_hr_procedures(body, sender, embed_model)
                    elif intent == "ADMIN":
                        reply_text = handle_administrative_procedures(body, sender)
                    elif intent == "INQUIRY":
                        reply_text = handle_general_inquiries(body, sender, kb)
                    elif intent == "User_Requests":
                        reply_text = handle_User_Requests(body, sender, kb, attachments, processor, embed_model)
                    elif intent == "ADMIN_2":
                        reply_text = handle_administrative_procedures_2(body, sender, kb, attachments, processor, embed_model)
                    # ج. الرد الافتراضي باستخدام RAG في حال عدم انطباق الشروط أعلاه
                    if not reply_text:
                        fixed_ctx = get_fixed_context(body, kb)
                        reply_text = rag_answer_email(body, fixed_ctx)

                # د. إنشاء المسودة وتوثيقها للتقييم (الميزة الجديدة)
                if reply_text:
                    # إنشاء المسودة في Gmail
                    draft_success = create_draft(sender, subject, reply_text, reply_to_id=msg_id)
                    
                    if draft_success:
                        # توثيق الرد في ملف الإكسل للمقارنة بـ Ground Truth لاحقاً
                        log_draft_for_evaluation(msg_id, sender, intent, body, reply_text)
                
                # هـ. تنظيف الملفات المؤقتة
                if attachments:
                    for att in attachments:
                        if os.path.exists(att): os.remove(att)

            # 2. مزامنة الإيميلات المرسلة والتعلم من ردود المدير البشرية
            did_learn_something_new = sync_sent_emails_to_history()
            
            # تحديث القاعدة المعرفية إذا تم استنتاج قاعدة إدارية جديدة
            if did_learn_something_new:
                print("♻️ تم اكتشاف قواعد جديدة.. جاري تحديث الدماغ (Knowledge Base)...")
                kb = initialize_fixed_knowledge(processor, embed_model)
                print("✅ تم تحديث الدماغ بنجاح.")
            
        except Exception as e:
            print(f"⚠ خطأ غير متوقع في الحلقة الرئيسية: {e}")
            time.sleep(5)
        
        # الانتظار قبل الفحص التالي
        time.sleep(10)
if __name__ == "__main__":
    run_auto_bot()


/tmp/ipykernel_3739701/531374829.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(


🚀 جاري تشغيل بوت الرد الآلي المطور (نظام التقييم والتعلم الذاتي)...
✅ تم بناء القاعدة المعرفية: 1735 قطع كبيرة و 10150 قطع صغيرة.
✅ تم تفعيل النظام بالكامل. البوت في وضع الاستعداد...
📨 1 رسائل جديدة.

📩 إيميل جديد من: xmio.511x@gmail.com
🎯 النية المكتشفة: HR
⏳ جاري البحث عن قاعدة مناسبة لطلب HR: المكرم مدير قسم الاشراف

السلام عليكم ورحمة الله وبركاته:

تحية طيبة , اود الإفادة برغبتي بالتقدم بطلب اجازة بتاريخ 2026/1/8 م لمدة 5
أيام وحيث ان المهندس / سامي سيقوم بمهامي خلال هذه الفترة.

ارجو الموافقة على طلب الاجازة .

ولسعادتكم تحياتي ,,

مقدم الطلب :

 مهندس / خالد محمد الصبيح
...

🔍 [Advanced Rule Retrieval] Intent: HR
   MATCHED PARENT RULE: ...1. الهوية: مدير قسم الإشراف  
2. الحالة: عندما يُطلب موافقة إجازة من قبل موظف  
3. الإجراء والمخاطب: إبلاغ مدير إدارة الموارد البشرية بالطلب وإرساله للموافقة عليه  
4. هيكل الرد: ابدأ بـ "سعادة..."، أشر للطلب بـ "نود الإحاطة"، اختتم بـ "ولسعادتكم تحياتي"...
   -> RETRIEVED PARENT RULE ID: 3
   MATCHED PARENT RULE: ...1. **الهوية**: مدير قسم ال

/home/alotaime/services.py:238: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'سعادة مدير إدارة الموارد البشرية

السلام عليكم ورحمة الله وبركاته:

نود الاحاطة بطلب الموظف: مهندس / خالد محمد الصبيح اجازة اعتبارًا من
تاريخ 2026/1/8 م لمدة 5 أيام، وفي تلك الفترة سيقوم المهندس / سامي
بالإنابة عنه .

ارجو اكمال اللازم وفي حال الموافقة على ذلك يتم تزويدنا بها ليتسنى لنا
ابلاغ الموظف بذلك.

ولسعادتكم تحياتي

مدير قسم الاشراف
' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 'Ground Truth'] = final_body


📊 تم تحديث Ground Truth للإيميل المرتبط بـ: CAN=DwnuvKQ6BPjUvH41SZ29WPr4K=uk32Ts=6wtPu8Yupr=WPQ@mail.gmail.com
🎓 تم اكتشاف رد بشري على معاملة HR (ID: <CANqVMQ1+TV+soD=u_CE+pU+Pdgstnh4Gq0Av19zWgAAh9PvZsQ@mail.gmail.com>)... جاري التعلم.
✅ تم تعلم قاعدة جديدة وحفظها في: HR_rules.txt
♻️ تم اكتشاف قواعد جديدة.. جاري تحديث الدماغ (Knowledge Base)...
✅ تم بناء القاعدة المعرفية: 1735 قطع كبيرة و 10150 قطع صغيرة.
✅ تم تحديث الدماغ بنجاح.
📨 1 رسائل جديدة.

📩 إيميل جديد من: xmio.511x@gmail.com
🎯 النية المكتشفة: HR
⏳ جاري البحث عن قاعدة مناسبة لطلب HR: المكرم مدير قسم الاشراف

السلام عليكم ورحمة الله وبركاته:

نظرا لحصول ظروف شخصية اريد التقدم بطلب  اجازة من تاريخ  20/2/2026م لمدة شهر
وحيث ان المهندس / فيصل سيقوم بمهامي خلال هذه الفترة.

ارجو الموافقة على طلب الاجازة .

ولسعادتكم تحياتي ,,

مقدم الطلب :

 موظف / محمد السبيعي
...

🔍 [Advanced Rule Retrieval] Intent: HR
   MATCHED PARENT RULE: ...1. الهوية: مدير قسم الإشراف  
2. الحالة: عندما يُطلب موافقة إجازة من قبل موظف  
3. الإجراء والمخاطب: إب

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🔍 [Advanced Rule Retrieval] Intent: HR
   MATCHED PARENT RULE: ...1. الهوية: مدير قسم الإشراف  
2. الحالة: عندما يُطلب موافقة إجازة من قبل موظف  
3. الإجراء والمخاطب: إبلاغ مدير إدارة الموارد البشرية بالطلب وإرساله للموافقة عليه  
4. هيكل الرد: ابدأ بـ "سعادة..."، أشر للطلب بـ "نود الإحاطة"، اختتم بـ "ولسعادتكم تحياتي"...
   -> RETRIEVED PARENT RULE ID: 3
   MATCHED PARENT RULE: ...1. **الهوية**: مدير قسم الإشراف  
2. **الحالة**: عند تلقي طلب إجازة من موظف  
3. **الإجراء والمخاطب**: إبلاغ مدير الموارد البشرية بالطلب مع ذكر فترة الإجازة والشخص البديل، وإرساله للمتابعة أو إبداء الرأي  
4. **هيكل الرد**:  
   - يبدأ بـ "سعادة..."  
   - يستخدم عبارة "نود الإحاطة بأن..." لذكر تفاصيل الطلب  
   - يشير إلى الحاجة إلى متابعة أو رأي من المخاطب  
   - يُغلق الرد بعبارة "ولسعادتكم تحياتي"...
   -> RETRIEVED PARENT RULE ID: 4
   MATCHED PARENT RULE: ...1. **الهوية**: مدير قسم الإشراف  
2. **الحالة**: عندما يُقدّم موظف طلبًا لإجازة ولا يوجد بديل عنه خلال فترة الغياب  
3. **الإجراء والمخاطب**: رفض